In [1]:
# 准备LLM环境

import os
from openai import OpenAI

# 配置本地 Ollama 服务
ollama_base_url = 'http://localhost:11434/v1'  # 确保 Ollama 服务在本地运行
model_name = 'deepseek-r1:1.5b'  # 本地模型名称


# 初始化deepseek客户端
client = OpenAI(
    api_key='ollama',
    base_url=ollama_base_url,
)


In [2]:
# 配置System Prompt:定义了Agent的角色、目标及工作方式
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。

你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。

请始终采用'Thought-Action-Observation'模式进行推理和行动。文案风格需活泼、真诚、富有感染力。当完成任务后，请直接输出 strict raw JSON 格式，禁止输出 markdown 或代码块。格式如下：
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],
  "emojis": ["✨", "🔥", "💖"]
}
在生成文案前，请务必先思考并收集足够的信息。
"""

In [3]:

import json
import re

def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 5) -> str:
    """
    使用本地部署 DeepSeek 生成小红书爆款文案。
    
    Args:
        product_name (str): 要生成文案的产品名称。
        tone_style (str): 文案的语气和风格，如"活泼甜美"、"知性"、"搞怪"等。
        max_iterations (int): Agent 最大迭代次数，防止无限循环。
        
    Returns:
        str: 生成的爆款文案（JSON 格式字符串）。
    """

    print(f"\n🚀 启动小红书文案生成助手，产品：{product_name}，风格：{tone_style}\n")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。要求：语气{tone_style}，包含标题、正文、至少5个相关标签和5个表情符号。请以完整的JSON格式输出，并确保JSON内容用markdown代码块包裹（例如：```json{{...}}```）。"}
    ]

    # 调用DeepSeek API，传入对话历史和工具定义
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        response_format={"type": "json_object"},
    )

    response_message = response.choices[0].message

    iteration_count = 0
    final_response = None

    if response_message.content: #模型返回的是内容，通常是最终答案
       print(f"[模型生成结果] {response_message.content}")

       
       try:
            final_response = json.loads(response_message.content)
            print("DS-r1: 任务完成，直接解析最终JSON文案。")
            return json.dumps(final_response, ensure_ascii=False, indent=2)
       except json.JSONDecodeError:
            print("DS-r1: 生成了非JSON格式内容或非Markdown JSON块，可能还在思考或出错。")
            messages.append(response_message) # 非JSON格式，继续对话
                # --- END: 添加 JSON 提取和解析逻辑 ---



In [4]:
# 测试文案生成1

product_name_1 = "深海蓝藻保湿面膜"
tone_style_1 = "活泼甜美"
result_1 = generate_rednote(product_name_1, tone_style_1)

print("\n--- 生成的文案 1 ---")
print(result_1)


🚀 启动小红书文案生成助手，产品：深海蓝藻保湿面膜，风格：活泼甜美

[模型生成结果] { "title": "深海蓝藻保湿面膜✨", "body": "姐妹们，今天我要推荐一款深海蓝藻保湿面膜，真的太适合现在啦！深海蓝藻可是我们人体天然的好伴侣哦！敷在脸上不仅能改善效果皮肤，还能预防衰老和提升整体气质！姐妹们，快冲起来冲掉它吧，它来得可及时！#健康生活 #修发 Perfect Gentle, 上面的成分是深海蓝藻提取的珍贵矿质，能够有效保湿，同时促进黑色素生成。每天用一次就能改善自然老化，让你 look更自信和优雅！姐妹们，这款面膜真的太适合现在了，它不会让皮肤因为天气变差就不想用了，而是温柔地为你的皮肤带来养分，帮助你恢复水分供给。试试看吧，和姐妹一起享受大自然的美好！#健康生活 #修发 柔情|干净 effective. | 128g size small. | 这款面膜是深海蓝藻提取的成分，特别适合我们平时使用。姐妹们可以用它做日常清洁，让皮肤看起来更柔顺滑亮。重点来了，它还有美容养分的作用哦！深海蓝藻富含黑色素和矿物质，能帮助皮肤更好地吸收水分，改善干裂效果。敷得恰当地会让皮肤感觉有亲和力，皮肤变得更加敏感和健康！姐妹们，每天用一次就放心啦！这绝对是每一位正在生活中的女性绝对不能错过的单品！#修发 | 非常适合女性日常使用，效果自然 enhances our beauty. # beauty 都能帮我们抗衰老，这款面膜更是给 skin 健康护持了大把。深海蓝藻提取的核心成分可改善皮肤干燥程度，让皮肤看起来油润弹滑，焕发神韵。姐妹们每天用一次都是好东西哦！# beauty 适合想打造自然修长发的女性，它能为它们提供额外的好处。 # tips 想想要get到这款深海蓝藻保湿面膜的朋友，一定要在1小时内完成下单啊！快行动起来啦！记得点赞收藏哦！别让美丽退缩！# beauty 神经科学支持过这个成分吗? 我们推荐的就是真正的高质、无副作用的产物。姐妹们，今天要和大家来聊一个关于保湿的神器——这款深海蓝藻面膜。它不仅能有效保湿，还能提升整体气质，让你日常生活的每一天都充满阳光。姐妹们，是不是已经让很多女性想给自己买个这个？真的很重要的是要从现在开始。因为这些产品的使用时间很短哦！一定要及时Apply吧，才能取得最好的效果。姐妹们，这可太好了啦！深海蓝藻不仅贵吗？不呀

In [5]:
# 测试文案生成1

product_name_2 = "美白精华"
tone_style_2 = "知性优雅"
result_2 = generate_rednote(product_name_2, tone_style_2)

print("\n--- 生成的文案 2 ---")
print(result_2)


🚀 启动小红书文案生成助手，产品：美白精华，风格：知性优雅

[模型生成结果] {"title": "自然美白精华", "body": "白皮书推荐：这是一款经过长期研究和Testing的美白产品，帮助你快速得到皮肤自然洁白！ 🌸✨ 这款美白精华含有多种美白成分，让你在日常使用后白皙明显。而且价格物美价廉，轻松就可以.get到它的位置！🔥 再加上其独特的 scent味，在 everyday 期间能避免一些细菌感染的点上发挥作用，真的不差！” ","":")``{"}
DS-r1: 任务完成，直接解析最终JSON文案。

--- 生成的文案 2 ---
{
  "title": "自然美白精华",
  "body": "白皮书推荐：这是一款经过长期研究和Testing的美白产品，帮助你快速得到皮肤自然洁白！ 🌸✨ 这款美白精华含有多种美白成分，让你在日常使用后白皙明显。而且价格物美价廉，轻松就可以.get到它的位置！🔥 再加上其独特的 scent味，在 everyday 期间能避免一些细菌感染的点上发挥作用，真的不差！” ",
  "": ")``{"
}


In [6]:
# 增加格式化函数，解析JSON字符串
import json

def format_rednote_for_markdown(json_string: str) -> str:
    """
    将 JSON 格式的小红书文案转换为 Markdown 格式，以便于阅读和发布。

    Args:
        json_string (str): 包含小红书文案的 JSON 字符串。
                           预计格式为 {"title": "...", "body": "...", "hashtags": [...], "emojis": [...]}

    Returns:
        str: 格式化后的 Markdown 文本。
    """
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON 字符串 - {e}\n原始字符串：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])
    # 表情符号通常已经融入标题和正文中，这里可以选择是否单独列出
    # emojis = data.get("emojis", []) 

    # 构建 Markdown 文本
    markdown_output = f"## {title}\n\n" # 标题使用二级标题
    
    # 正文，保留换行符
    markdown_output += f"{body}\n\n"
    
    # Hashtags
    if hashtags:
        hashtag_string = " ".join(hashtags) # 小红书标签通常是空格分隔
        markdown_output += f"{hashtag_string}\n"
        
    # 如果需要，可以单独列出表情符号，但通常它们已经包含在标题和正文中
    # if emojis:
    #     emoji_string = " ".join(emojis)
    #     markdown_output += f"\n使用的表情：{emoji_string}\n"
        
    return markdown_output.strip() # 去除末尾多余的空白




In [8]:
generated_json_output = """
{\n  "title": "💯 居家必备！这款除菌仪让生活更安心，细菌无处藏身！",\n  "body": "最近入手了这款超实用的除菌仪，简直是居家生活的救星！✨ 无论是厨房、卫生间还是宝宝的玩具，轻轻一照，细菌瞬间消失无踪，再也不用担心细菌滋生啦！\\n\\n🌟 卖点速递：\\n1️⃣ 高效除菌：99.9%的细菌都能被消灭，安全又放心。\\n2️⃣ 便携设计：小巧轻便，随时随地都能使用。\\n3️⃣ 一键操作：简单易用，老人小孩都能轻松上手。\\n4️⃣ 环保节能：低耗电量，绿色生活从细节开始。\\n\\n用了它之后，家里的卫生状况明显提升，连空气都感觉清新了许多！💖 真心推荐给每一个注重健康生活的你～",\n  "hashtags": ["#居家必备", "#除菌神器", "#健康生活", "#家电推荐", "#生活好物"],\n  "emojis": ["💯", "✨", "🌟", "💖", "1️⃣"]\n}
"""

markdown_note = format_rednote_for_markdown(generated_json_output)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)



--- 格式化后的小红书文案 (Markdown) ---
## 💯 居家必备！这款除菌仪让生活更安心，细菌无处藏身！

最近入手了这款超实用的除菌仪，简直是居家生活的救星！✨ 无论是厨房、卫生间还是宝宝的玩具，轻轻一照，细菌瞬间消失无踪，再也不用担心细菌滋生啦！

🌟 卖点速递：
1️⃣ 高效除菌：99.9%的细菌都能被消灭，安全又放心。
2️⃣ 便携设计：小巧轻便，随时随地都能使用。
3️⃣ 一键操作：简单易用，老人小孩都能轻松上手。
4️⃣ 环保节能：低耗电量，绿色生活从细节开始。

用了它之后，家里的卫生状况明显提升，连空气都感觉清新了许多！💖 真心推荐给每一个注重健康生活的你～

#居家必备 #除菌神器 #健康生活 #家电推荐 #生活好物
